# Scribal Identification: Clustering & Visualization

Unsupervised clustering of deed embeddings from a ViT-Small backbone pretrained
on our own corpus of deeds. The pipeline:

1. **Load** pre-extracted VLAD embeddings (38 400-d per document)
2. **PCA**: 50 dimensions (variance retention, denoising)
3. **UMAP**: 2 dimensions (neighbourhood-preserving projection)
4. **HDBSCAN**: density-based clusters with soft membership probabilities
5. **Medoids**: most representative document ("medoid") per cluster
6. **Visualize**: with Bokeh (interactive scatter + Voronoi overlay, medoids highlighted)

## 1 — Configuration

In [1]:
import numpy as np
import base64, re, json, os, io
from pathlib import Path
from PIL import Image

from sklearn.decomposition import PCA
from sklearn.cluster import HDBSCAN
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.spatial import Voronoi

from bokeh.plotting import figure, output_file, save
from bokeh.models import ColumnDataSource, HoverTool, Label
from bokeh.palettes import Category20

In [2]:
# ── Paths & data ────────────────────────────────────────────────────
EMBEDDINGS_FILE = "../data/embeddings/embeddings_20-03-2026.npz"
IMAGES_DIR      = "../data/deeds-img"
THUMBNAIL_SIZE  = (1000, 1000)
THUMBNAIL_CACHE = "../outputs/thumbnail_cache.json"
UMAP_FILENAME = "../outputs/umap_hdbscan.html"

# ── Dimensionality reduction ────────────────────────────────────────
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST    = 0.1

# ── HDBSCAN ─────────────────────────────────────────────────────────
HDBSCAN_MIN_CLUSTER_SIZE = 10
HDBSCAN_MIN_SAMPLES      = 5
HDBSCAN_METHOD            = "leaf"   # use "eom" for coarser clusters
HDBSCAN_EPSILON           = 0.0      # >0 prevents merging subclusters beyond this dist

# ── Misc ────────────────────────────────────────────────────────────
DISTANCE_METRIC = "euclidean"
RANDOM_SEED     = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 2 — Load embeddings & apply exclusions

Each document is represented by a single 38 400-d VLAD vector (100 clusters × 384-d
ViT-Small descriptors), power- and L2-normalised. Documents listed in `exclude.txt`
(e.g. heavily damaged deeds) are dropped before any analysis.

In [3]:
data       = np.load(EMBEDDINGS_FILE)
embeddings = data["embeddings"]
filenames  = data["filenames"]

exclude = set(line.strip() for line in open("../data/exclude.txt"))
keep    = np.array([not any(e in str(fn) for e in exclude) for fn in filenames])

embeddings = embeddings[keep]
filenames  = filenames[keep]
N = len(embeddings)

print(f"Excluded {(~keep).sum()} docs > {N} embeddings retained")

Excluded 8 docs > 462 embeddings retained


## 3 — Map embeddings to on-disk image files

Embedding filenames use a `0-<ID>_…` scheme; the raw images may or may not
carry the `0-` prefix. We normalise both to a bare numeric ID for matching.

In [4]:
images_dir  = Path(IMAGES_DIR)
actual_files = [f for f in images_dir.iterdir() if f.is_file()]


def _extract_id(name: str, prefix: str = r"(?:0-)?") -> str | None:
    """Return the bare numeric ID from an embedding or disk filename."""
    m = re.match(rf"^{prefix}(\d+)_", name)
    return m.group(1).lstrip("0") or "0" if m else None


disk_by_id = {_extract_id(f.name): f for f in actual_files if _extract_id(f.name)}
print(f"{len(actual_files)} images on disk:  {len(disk_by_id)} indexed by ID")

470 images on disk:  470 indexed by ID


## 4 — PCA (38 400 → 50)

Aggressive reduction before UMAP. 50 components is a practical sweet spot
that retains the dominant variance while cutting noise and speeding up the
downstream distance computation.

In [5]:
N_PCA = 50
pca = PCA(n_components=N_PCA, random_state=RANDOM_SEED)
embeddings_pca = pca.fit_transform(embeddings)

print(f"Explained variance ({N_PCA} PCs): {pca.explained_variance_ratio_.sum():.2%}")

Explained variance (50 PCs): 59.11%


## 5 — Pairwise distance matrix

Computed on the 50-d PCA space. The full N × N matrix is passed to UMAP as a
precomputed distance matrix, which gives us explicit control over the metric.

In [6]:
D_emb = squareform(pdist(embeddings_pca, metric=DISTANCE_METRIC))
print(f"Distance matrix: {D_emb.shape}  ({DISTANCE_METRIC})")

Distance matrix: (462, 462)  (euclidean)


## 6 — UMAP (50 → 2)

Non-linear projection that preserves local neighbourhood structure.
We feed the precomputed distance matrix so the UMAP metric matches our PCA metric.

In [7]:
from umap import UMAP as UMAPReducer

reducer = UMAPReducer(
    n_components=2,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric="precomputed",
    random_state=RANDOM_SEED,
)
embeddings_2d = reducer.fit_transform(D_emb)
print(f"UMAP output: {embeddings_2d.shape}")

/Users/mikekestemont/miniconda3/envs/bayes/lib/python3.10/site-packages/umap/umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
/Users/mikekestemont/miniconda3/envs/bayes/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


UMAP output: (462, 2)


## 7 — HDBSCAN parameter sweep

Quick grid search over `min_cluster_size` (mcs) and `min_samples` (ms) to inspect
the stability of the cluster count. Each cell shows `<clusters>c / <noise%>`.

In [8]:
mcs_vals = [4, 5, 6, 8, 10, 12, 15, 20]
ms_vals  = [3, 5, 7, 10]

rows = []
for mcs in mcs_vals:
    row = {"mcs": mcs}
    for ms in ms_vals:
        labs = HDBSCAN(
            min_cluster_size=mcs, min_samples=ms,
            metric="euclidean", cluster_selection_method="eom",
            cluster_selection_epsilon=HDBSCAN_EPSILON,
        ).fit_predict(embeddings_2d)
        nc = labs.max() + 1
        nn = (labs == -1).sum()
        row[f"ms={ms}"] = f"{nc}c / {nn/len(labs)*100:.1f}%"
    rows.append(row)

# Pretty-print as a simple table
header = f"{'mcs':>5}" + "".join(f"  {k:>12}" for k in rows[0] if k != "mcs")
print(header)
print("-" * len(header))
for r in rows:
    line = f"{r['mcs']:>5}"
    for k, v in r.items():
        if k != "mcs":
            line += f"  {v:>12}"
    print(line)

  mcs          ms=3          ms=5          ms=7         ms=10
-------------------------------------------------------------
    4    43c / 5.0%   30c / 11.9%   23c / 13.4%   17c / 13.0%
    5    33c / 5.0%   29c / 10.8%   23c / 13.4%   17c / 13.0%
    6    29c / 7.1%   26c / 10.6%   22c / 12.1%   17c / 13.0%
    8    21c / 6.1%    20c / 8.4%    18c / 9.1%   17c / 13.0%
   10    17c / 6.5%    18c / 6.9%    18c / 9.1%   17c / 13.0%
   12    16c / 6.5%    15c / 5.6%    16c / 8.4%   15c / 12.6%
   15    13c / 8.7%    14c / 8.2%     8c / 5.0%     8c / 5.8%
   20   12c / 12.6%     7c / 8.2%     7c / 8.9%     6c / 9.7%


## 8 — HDBSCAN clustering (final)

Run with the chosen parameters on the 2-D UMAP embedding. `leaf` mode tends
to produce more fine-grained subclusters — appropriate here because adjacent
subclusters may represent the *same* scribe at different career stages and we
prefer to validate merges through the human annotation campaign rather than
commit to them automatically.

In [9]:
eps_label = f", eps={HDBSCAN_EPSILON}" if HDBSCAN_EPSILON > 0 else ""
print(f"HDBSCAN: mcs={HDBSCAN_MIN_CLUSTER_SIZE}, ms={HDBSCAN_MIN_SAMPLES}, "
      f"method={HDBSCAN_METHOD}{eps_label}")

clusterer = HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples=HDBSCAN_MIN_SAMPLES,
    metric="euclidean",
    cluster_selection_method=HDBSCAN_METHOD,
    cluster_selection_epsilon=HDBSCAN_EPSILON,
    store_centers="medoid",
)
hdb_labels = clusterer.fit_predict(embeddings_2d)
hdb_probs  = clusterer.probabilities_

n_clusters = hdb_labels.max() + 1
n_noise    = (hdb_labels == -1).sum()
print(f"→ {n_clusters} clusters  +  {n_noise} noise ({n_noise/N:.1%})")
print()

for ci in range(n_clusters):
    print(f"  H{ci:>2}: {(hdb_labels == ci).sum():>3} docs")

HDBSCAN: mcs=10, ms=5, method=leaf
→ 19 clusters  +  59 noise (12.8%)

  H 0:  18 docs
  H 1:  21 docs
  H 2:  12 docs
  H 3:  25 docs
  H 4:  36 docs
  H 5:  33 docs
  H 6:  39 docs
  H 7:  11 docs
  H 8:  21 docs
  H 9:  30 docs
  H10:  11 docs
  H11:  16 docs
  H12:  30 docs
  H13:  26 docs
  H14:  19 docs
  H15:  12 docs
  H16:  20 docs
  H17:  13 docs
  H18:  10 docs


## 9 — Medoid identification

For each cluster, select the single document closest to the cluster centroid
among those with near-maximal membership probability. These medoids serve as
the visual "prototypes" for each cluster in the paper.

In [10]:
medoid_indices = []
for ci in range(n_clusters):
    members  = np.where(hdb_labels == ci)[0]
    probs    = hdb_probs[members]
    # restrict to top-probability members (within 0.01 of the max)
    top      = members[probs >= probs.max() - 0.01]
    centroid = embeddings_2d[members].mean(axis=0, keepdims=True)
    dists    = cdist(embeddings_2d[top], centroid).flatten()
    medoid   = top[np.argmin(dists)]
    medoid_indices.append(medoid)
    print(f"  H{ci}: {filenames[medoid]}  (prob {hdb_probs[medoid]:.3f})")

  H0: 0-0096_06_10_1347-01.png  (prob 1.000)
  H1: 0-0128_08_08_1343-01.png  (prob 1.000)
  H2: 0-0184_13_01_1344-01.png  (prob 1.000)
  H3: 0-0077_05_09_1336-01.png  (prob 1.000)
  H4: 0-0381_26_06_1350-01.png  (prob 1.000)
  H5: 0-0095_06_10_1334-01.png  (prob 1.000)
  H6: 0-0193_13_08_1306-01.png  (prob 1.000)
  H7: 0-0116_07_12_1352-01.png  (prob 1.000)
  H8: 0-0316_20_09_1348-01.png  (prob 1.000)
  H9: 0-0424_30_01_1351-01.png  (prob 1.000)
  H10: 0-0112_07_05_1343-01.png  (prob 1.000)
  H11: 0-0028_01_10_1354-01.png  (prob 1.000)
  H12: 0-0021_01_09_1331-01.png  (prob 1.000)
  H13: 0-0453_05_06_1315-01.png  (prob 1.000)
  H14: 0-0141_09_06_1332-01.png  (prob 1.000)
  H15: 0-0243_16_07_1335-01.png  (prob 1.000)
  H16: 0-0143_09_07_1336-01.png  (prob 1.000)
  H17: 0-0394_27_05_1347-01.png  (prob 1.000)
  H18: 0-0234_15_05_1344-01.png  (prob 1.000)


## 10 — Thumbnail generation (with disk cache)

Base64-encoded PNG thumbnails are embedded in the Bokeh HTML so that
hovering over a point shows the deed image. A JSON cache avoids
re-encoding on repeated runs.

In [11]:
if Path(THUMBNAIL_CACHE).exists():
    with open(THUMBNAIL_CACHE) as f:
        thumb_cache = json.load(f)
else:
    thumb_cache = {}

thumbnails  = []
matched     = 0
cache_hits  = 0
cache_dirty = False

for fn in filenames:
    emb_id = _extract_id(str(fn), prefix=r"0-")
    if not emb_id:
        thumbnails.append(""); continue

    if emb_id in thumb_cache:
        thumbnails.append(thumb_cache[emb_id])
        matched += 1; cache_hits += 1; continue

    img_path = disk_by_id.get(emb_id)
    if img_path and img_path.exists():
        img = Image.open(img_path)
        img.thumbnail(THUMBNAIL_SIZE)
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        b64 = f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"
        thumbnails.append(b64)
        thumb_cache[emb_id] = b64
        cache_dirty = True
        matched += 1
    else:
        thumbnails.append("")

if cache_dirty:
    with open(THUMBNAIL_CACHE, "w") as f:
        json.dump(thumb_cache, f)

print(f"Thumbnails: {matched}/{N} matched  ({cache_hits} cached, "
      f"{matched - cache_hits} new)")

Thumbnails: 462/462 matched  (462 cached, 0 new)


## 11 — Interactive UMAP scatter plot (Bokeh)

Three glyph layers: noise points (grey diamonds), clustered points (coloured
circles sized by membership probability), and medoid stars. A Voronoi diagram
over the cluster centroids is drawn as dashed lines to delineate the regions.

In [12]:
palette = Category20[20]

hdb_colors = ["#888888" if l == -1 else palette[l % 20] for l in hdb_labels]
hdb_sizes  = [10 if l == -1 else 6 + p * 8
              for l, p in zip(hdb_labels, hdb_probs)]

output_file(UMAP_FILENAME)

# Split indices
noise_idx = np.where(hdb_labels == -1)[0]
clust_idx = np.where(hdb_labels != -1)[0]

# ── Data sources ────────────────────────────────────────────────────
source_noise = ColumnDataSource(dict(
    x=embeddings_2d[noise_idx, 0], y=embeddings_2d[noise_idx, 1],
    filename=[str(filenames[i]) for i in noise_idx],
    thumbnail=[thumbnails[i] for i in noise_idx],
    hdb_prob=[f"{hdb_probs[i]:.2f}" for i in noise_idx],
))

source_clustered = ColumnDataSource(dict(
    x=embeddings_2d[clust_idx, 0], y=embeddings_2d[clust_idx, 1],
    filename=[str(filenames[i]) for i in clust_idx],
    thumbnail=[thumbnails[i] for i in clust_idx],
    hdb_cluster=[str(hdb_labels[i]) for i in clust_idx],
    hdb_prob=[f"{hdb_probs[i]:.2f}" for i in clust_idx],
    color=[hdb_colors[i] for i in clust_idx],
    size=[hdb_sizes[i] for i in clust_idx],
))

source_medoids = ColumnDataSource(dict(
    x=embeddings_2d[medoid_indices, 0], y=embeddings_2d[medoid_indices, 1],
    filename=[str(filenames[i]) for i in medoid_indices],
    thumbnail=[thumbnails[i] for i in medoid_indices],
    hdb_cluster=[str(hdb_labels[i]) for i in medoid_indices],
    hdb_prob=[f"{hdb_probs[i]:.2f}" for i in medoid_indices],
    color=[hdb_colors[i] for i in medoid_indices],
    label=[f"H{i}" for i in range(n_clusters)],
))

# ── Figure ──────────────────────────────────────────────────────────
eps_tag = f", eps={HDBSCAN_EPSILON}" if HDBSCAN_EPSILON > 0 else ""
p = figure(
    title=(f"UMAP(n={UMAP_N_NEIGHBORS}, d={UMAP_MIN_DIST}) → "
           f"HDBSCAN(mcs={HDBSCAN_MIN_CLUSTER_SIZE}, ms={HDBSCAN_MIN_SAMPLES}, "
           f"{HDBSCAN_METHOD}{eps_tag}) — {n_clusters} clusters, {n_noise} noise"),
    width=1400, height=1000,
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

### 11a — Voronoi overlay

Dashed lines show the Voronoi diagram of cluster centroids, giving a visual
sense of which region of the UMAP space "belongs" to each cluster. Infinite
ridges are extended outward from the centroid barycentre.

In [13]:
all_centroids = np.array([
    embeddings_2d[hdb_labels == ci].mean(axis=0) for ci in range(n_clusters)
])

if n_clusters >= 3:
    vor = Voronoi(all_centroids)

    # Finite ridges
    for v0, v1 in vor.ridge_vertices:
        if v0 >= 0 and v1 >= 0:
            p.line(
                [vor.vertices[v0, 0], vor.vertices[v1, 0]],
                [vor.vertices[v0, 1], vor.vertices[v1, 1]],
                line_color="#cccccc", line_width=1.5,
                line_dash="dashed", line_alpha=0.9,
            )

    # Infinite ridges — project outward from the centre of mass
    x_span = np.ptp(embeddings_2d[:, 0]) + 2
    y_span = np.ptp(embeddings_2d[:, 1]) + 2
    center = all_centroids.mean(axis=0)

    for pts, (v0, v1) in zip(vor.ridge_points, vor.ridge_vertices):
        if v0 >= 0 and v1 >= 0:
            continue
        finite_v = v0 if v0 >= 0 else v1
        origin   = vor.vertices[finite_v]
        midpoint = all_centroids[pts].mean(axis=0)
        tangent  = all_centroids[pts[1]] - all_centroids[pts[0]]
        normal   = np.array([-tangent[1], tangent[0]])
        normal  /= np.linalg.norm(normal)
        if np.dot(midpoint - center, normal) < 0:
            normal = -normal
        far = origin + normal * max(x_span, y_span) * 2
        p.line([origin[0], far[0]], [origin[1], far[1]],
               line_color="#cccccc", line_width=1.5,
               line_dash="dashed", line_alpha=0.7)

elif n_clusters == 2:
    mid     = all_centroids.mean(axis=0)
    tangent = all_centroids[1] - all_centroids[0]
    normal  = np.array([-tangent[1], tangent[0]])
    normal /= np.linalg.norm(normal)
    span    = max(np.ptp(embeddings_2d[:, 0]), np.ptp(embeddings_2d[:, 1])) * 2
    p.line([mid[0] - normal[0]*span, mid[0] + normal[0]*span],
           [mid[1] - normal[1]*span, mid[1] + normal[1]*span],
           line_color="#cccccc", line_width=1.5,
           line_dash="dashed", line_alpha=0.7)

print(f"Voronoi overlay: {n_clusters} centroids")

Voronoi overlay: 19 centroids


### 11b — Scatter glyphs & hover tooltips

In [14]:
HOVER_HTML = '''<div>
  <div><strong>@filename</strong></div>
  <div>{extra}</div>
  <div><img src="@thumbnail"
       style="max-width:1000px; max-height:1000px;"></div>
</div>'''

# Noise diamonds
r_noise = p.scatter("x", "y", source=source_noise,
                    marker="diamond", size=10, alpha=0.55,
                    color="#888888", line_color="#555555", line_width=1,
                    legend_label="noise")

# Clustered circles
r_clust = p.scatter("x", "y", source=source_clustered,
                    size="size", alpha=0.6, color="color",
                    legend_field="hdb_cluster")

# Medoid stars
r_med = p.scatter("x", "y", source=source_medoids,
                  marker="star", size=25, alpha=1.0, color="color",
                  line_color="black", line_width=2,
                  legend_label="MEDOIDS")

p.add_tools(HoverTool(renderers=[r_noise], tooltips=HOVER_HTML.format(
    extra="<em>noise</em> · prob @hdb_prob")))
p.add_tools(HoverTool(renderers=[r_clust], tooltips=HOVER_HTML.format(
    extra="H@hdb_cluster · prob @hdb_prob")))
p.add_tools(HoverTool(renderers=[r_med],   tooltips=HOVER_HTML.format(
    extra="<strong>MEDOID @label</strong> · H@hdb_cluster · prob @hdb_prob")))

# Cluster labels next to medoids
for i, idx in enumerate(medoid_indices):
    p.add_layout(Label(x=embeddings_2d[idx, 0], y=embeddings_2d[idx, 1],
                       text=f" H{i}", text_font_size="11pt",
                       text_font_style="bold", text_color="black"))

p.legend.visible = False
p.xaxis.axis_label = "UMAP 1"
p.yaxis.axis_label = "UMAP 2"
save(p)

print(f"Saved {UMAP_FILENAME} ({Path(UMAP_FILENAME).stat().st_size/1024/1024:.1f} MB)")

Saved ../outputs/umap_hdbscan.html (474.1 MB)
